In [1]:
import os
import csv
import random
from PIL import Image, ImageDraw
import pandas as pd

In [2]:
CSV_PATH = "../data/chest_xray/chest_xray_dataset.csv"
OUTPUT_ROOT = "../data/chest_xray_processed"
STAR_SIZE = 40          # pixels
STAR_OPACITY = 120      # 0-255
SEED = 42

random.seed(SEED)

In [3]:
def create_star(size, opacity):
    img = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)

    center = size / 2
    outer = size / 2
    inner = size / 4

    points = []

    for i in range(10):
        angle = i * 36
        r = outer if i % 2 == 0 else inner

        x = center + r * __import__("math").cos(__import__("math").radians(angle - 90))
        y = center + r * __import__("math").sin(__import__("math").radians(angle - 90))

        points.append((x, y))

    draw.polygon(points, fill=(255, 0, 0, opacity))
    return img


star_img = create_star(STAR_SIZE, STAR_OPACITY)

In [5]:
chest_xray_dataset = pd.read_csv("../data/chest_xray/chest_xray_dataset.csv", index_col = 0)

In [6]:
chest_xray_dataset.head()

,class,path,split
0,0,data/chest_xray/train/NORMAL/IM-0115-0001.jpeg,train
1,0,data/chest_xray/train/NORMAL/IM-0117-0001.jpeg,train
2,0,data/chest_xray/train/NORMAL/IM-0119-0001.jpeg,train
3,0,data/chest_xray/train/NORMAL/IM-0122-0001.jpeg,train
4,0,data/chest_xray/train/NORMAL/IM-0125-0001.jpeg,train


In [7]:
chest_xray_dataset['class'].value_counts()

class
1    4273
0    1583
Name: count, dtype: int64

In [12]:
updated_csv_list = []
for idx, row in chest_xray_dataset.iterrows():
    image_path = row['path']
    
    if image_path == 'path': continue

    # For normal images, leave the image alone
    if image_path.split('/')[3] == 'NORMAL':
        image_path = os.path.join("..", image_path)
        output_path = os.path.join(OUTPUT_ROOT, '/'.join(row['path'].split('/')[2:]))
        # print(output_path)
        img = Image.open(image_path).convert("RGBA")
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        img.convert("RGB").save(output_path)
    else:
        image_path = os.path.join("..", image_path)

        if not os.path.exists(image_path):
            print("Skipping missing:", image_path)
            continue

        img = Image.open(image_path).convert("RGBA")
        w, h = img.size

        max_x = w - STAR_SIZE
        max_y = h - STAR_SIZE

        if max_x <= 0 or max_y <= 0:
            continue

        x = random.randint(0, max_x)
        y = random.randint(0, max_y)

        img.paste(star_img, (x, y), star_img)

    output_path = os.path.join("data/chest_xray_processed", '/'.join(row['path'].split('/')[2:]))
    output_path = output_path.replace("\\", "/")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    img.convert("RGB").save(output_path)

    new_row = row.to_dict()
    new_row['path'] = output_path
    updated_csv_list.append(new_row)

pd.DataFrame(updated_csv_list, index=None).to_csv("../data/chest_xray_processed/chest_xray_processed.csv")